# Triaxial permeation, TWO-PISTON — the one permeation run

Analysis for `triaxial_permeation_two_pist.lmp`: a laterally periodic gel slab between a **feed** reservoir held
at `P_target + dP` and a **permeate** reservoir held at `P_target`, each closed by an NPT-piston (Marioni et al.,
J. Membr. Sci. 738 (2026) 124837, Eq. 3; the paper's Fig. 2B rotated by 90°).  There is **no sweep**: the whole
production run is one continuous constant-pressure drive at one `dP`, so every field is a **time evolution** over
the production frames (colour = timestep, cividis; bold black = final).  The Phase 1.5 **zero-flux reference**
(both pistons at `P_target`) is the dashed baseline.

Sections: profile evolutions (total / partial / network stress, density) with the reference baseline; the solvent
volume fractions (mass fraction / Voronoi / λ-calibrated Voronoi, reference vs steady state and their evolution, with the
**pore-pressure ramp feed → permeate as the calibration pressure**); reservoir
pressures **measured on the pistons (block-averaged) vs applied**; the flux `Q_perm` from the **slope of the
permeate bead count** over independent windows (standard error), with the `z_perm` slope and the legacy
block-averaged piston velocity beside it; the permeability `k = Q_perm L /(A dP)` with a CI.  All code is in
`scripts/lib/triaxial.py` (`load_permeation`, `fig_perm_*`).  Notes at the end (2026-09-16, flux estimator
and volume fractions 2026-09-23).

## Files (synced by the sync cell)

Into `flow_data_local/permeation/<RUN_ID>/` (cluster `output_files/…`, names `<name>_<DATANAME>_<INTERACTION>_<NSTEPS>`):

| file | content |
|---|---|
| `sigma{zz,xx,yy}_{polymer,solvent}[_ref]_…` | group partial stress profiles (kinetic term included) |
| `solvent_density_z[_ref]_…` | solvent number / mass density profiles |
| `piston_position_…`, `piston_velocity_…`, `piston_force_…`, `piston_force_avg[_ref]_…` | `[feed \| perm]` columns |
| `piston_pressure_…` | `P_feed`, `P_perm` measured (`F_fluid/(lx ly)`) and applied |
| `permeation_…` | block-averaged `z_feed z_perm F_feed F_perm P_feed P_perm Q_perm N_permeate` |
| `permeate_count_…` | bead count below the support (cross-check) |
| `pressure_feed_…`, `pressure_permeate_…` | reservoir virial pressures + densities |
| `strain_zz_…`, `gel_dimensions_{rg,bb}_…`, `box_dimensions_…`, `polymer_com_…`, `stress_aniso_…` | geometry / diagnostics |
| `disp_z_polymer_…` | polymer displacement profile (not used here) |

Into `flow_data_local/traj_files.nosync/`: `traj_ref_…` (box header + wall planes), `traj_stress_…`.

## 1 · Setup and computation

In [ ]:
import sys, importlib
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

LIB = Path('lib').resolve()            # scripts/lib: triaxial.py (all analysis code) + volfrac.py
if str(LIB) not in sys.path:
    sys.path.insert(0, str(LIB))
import triaxial as tri
tri = importlib.reload(tri)            # pick up edits to lib/triaxial.py without a kernel restart
tri.setup_style()
print('analysis code: ', LIB / 'triaxial.py')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
#  CONFIG -- the only cell to edit when switching runs
# ══════════════════════════════════════════════════════════════════════════
cfg = tri.Config(
    DATANAME    = "final_config_slab_support_periodic_5beads_tall_rho04_new_1.0_1.0_14000002_two_pist",
    INTERACTION = "1.0_1.0",          # epsSS_epsSP
    NSTEPS      = None,           # production steps = the <steps> tag; None resolves it from the files after the sync
    RUN_ID      = "two_pist_perm_1",  # local folder under flow_data_local/{permeation,plots}
    mode        = "permeation",   # NO levels: one continuous constant-dP drive
    two_pist    = True,           # Expanse folder triaxial_permeation_two_pist
    DP_PISTON   = None,           # applied dP; None -> read from piston_pressure (P_feed_app - P_perm_app)
    plateau_frac      = 0.25,     # trailing fraction used for the "steady" profile means
    plateau_frac_auto = 0.45,     # longest candidate window for the Q_perm steady-state plateau (drift test)
    P_BARO      = 1.5,            # P_target (normalises the total-stress panels)
    # ---- solvent volume fractions (lib/volfrac.py): mass fraction, Voronoi, lambda-calibrated Voronoi ----
    VOR_ENABLE = True, REF_VOR_FRAMES = 3, VOR_MAX_FRAMES = 4, VOR_EVO_FRAMES = 5,   # ~20 s per tessellated frame
    P_CAL = 1.5, P_CAL_MODE = 'pore',  # the P handed to lambda(phi_p, P) per z-bin.  'pore': the pore-pressure ramp from the
                                       # measured feed baseline to the measured permeate baseline across the membrane (p_pore
                                       # falls feed -> permeate under flow); 'const': P_CAL everywhere (the compression choice:
                                       # drained equilibrium); 'thermo': the local P_th = -1/3 tr sigma^t.  See Notes.
)

In [ ]:
SYNC, FORCE_SYNC = True, False
if SYNC:
    tri.sync_from_expanse(cfg, force=FORCE_SYNC)

In [ ]:
# R -- the zero-flux reference (Phase 1.5; both pistons at P_target): geometry, reference profiles, wall planes
# P -- the production run: stress / density evolutions, pistons, Q_perm(t), permeability
R = tri.load_reference(cfg)
P = tri.load_permeation(cfg, R)
tri.add_perm_volume_fractions(cfg, R, P)   # mass fraction / Voronoi / calibrated phi_s: tessellates ~9 frames (~20 s each)
tri.print_perm_summary(cfg, R, P)

## 2 · Figures

In [ ]:
# 1 · Wet pistons: (a) displacement of the feed and permeate pistons; (b) P measured (block-averaged F_fluid/A from
#     piston_force_avg) vs applied, reservoir virial pressures (pf cadence since 2026-09-23) dotted
tri.fig_perm_pistons(cfg, R, P);

In [ ]:
# 2 · Total stress / P_bath evolution σ^t_zz, σ^t_xx, σ^t_yy -- zero-flux reference dashed -> drive (cividis) -> final (bold)
tri.fig_total_stress(cfg, R, P);

In [ ]:
# 3 · Solvent and polymer partial σ_zz evolutions with the total superimposed
tri.fig_partial_stress(cfg, R, P);

In [ ]:
# 4 · Network stress evolution σ' = σ^t − p_pore (feed-reservoir baseline)
tri.fig_network_stress(cfg, R, P);

In [ ]:
# 5 · Solvent mass-density evolution with the zero-flux reference
tri.fig_perm_density(cfg, R, P);

In [ ]:
# 6 · Solvent volume fraction phi_s -- mass fraction, Voronoi and lambda-calibrated Voronoi: (a) zero-flux reference,
#     (b) steady permeation (trailing-window mean); (c) the calibration pressure P_local(z) handed to lambda(phi_p, P)
#     (P_CAL_MODE='pore': the pore-pressure ramp feed -> permeate across the membrane) and (d) the resulting lambda(z)
tri.fig_perm_volfrac(cfg, R, P);

In [ ]:
# 7 · phi_s evolution over the drive: mass fraction (every density snapshot), Voronoi and calibrated (tessellated frames);
#     zero-flux reference dashed
tri.fig_perm_volfrac_evolution(cfg, R, P);

In [ ]:
# 8 · Q_perm: (a) piston-velocity trace for context + the estimates; (b) N_permeate(t) with the independent-window
#     slopes that give the PRIMARY Q +/- SE (cfg.q_win_steps per window); z_perm slope and legacy block mean beside it
tri.fig_perm_flux(cfg, R, P);

In [ ]:
# 9 · Permeability k = Q_perm L/(A dP): N-slope Q with the applied and the measured dP (primary), z_perm-slope Q, legacy -- with CIs
tri.fig_perm_permeability(cfg, R, P);

In [ ]:
# 10 · Thermodynamic pressure P_th = −⅓ tr(σ^t) evolution (positive under compression; dotted = P_bath)
tri.fig_thermo_pressure(cfg, R, P);

In [ ]:
# 11 · Osmotic pressure Π = −⅓ tr(σ′): zero-flux reference and final (steady-flow) state, feed-reservoir baseline.
#     Under flow p_pore is not uniform across the gel, so the final curve's slope is mostly the pore-pressure drop
tri.fig_osmotic_pressure(cfg, R, P);


## Notes

* **Flux (2026-09-23)**.  The primary flux is the **slope of the permeate bead count**, `Q_perm = (dN_permeate/dt)/ρ_s,0`
  (`permeate_count_….dat`, solvent that crossed below the support; `ρ_s,0` = reference bulk density).  `N` is an
  integral of the flow, so its slope over long windows averages the piston's thermal jitter away; the
  block-averaged piston velocity `A·dz_perm/dt` (`permeation_….dat`, column `Q_perm`) does not — for a
  2.3×10⁷-mass sheet `v_thermal ≈ v_drift`, and the 5000-step blocks are shorter than the sheet's ~20k-step
  velocity correlation time, so its block scatter (~11–24 % of the mean in the 2026-09-22 run) is not an error
  bar.  The uncertainty is the **standard error over independent windows** of `cfg.q_win_steps` (150k) steps
  (t-interval), and `tri.plateau_window_slopes` picks the longest trailing fraction whose per-window slopes show
  no trend (first vs second half within 2 σ).  The `z_perm` slope over the same windows (`Q = −A·dz_perm/dt`)
  is the second estimate, the legacy block-bootstrap mean of the velocity trace the third; all three are drawn
  in figure 6 and carried into figure 7.
* **Volume fractions under flow (2026-09-23)**.  Figures 6–7 draw the three solvent-fraction estimators of the compression
  notebooks (`lib/volfrac.py`): the mass fraction `φ_s^mf = ρ_s/ρ_s,0` from the density profiles, the periodic Voronoi
  fraction `φ_s^vor` (voro++ on `traj_stress` frames; `VOR_EVO_FRAMES` evenly over the drive plus `VOR_MAX_FRAMES` inside
  the steady window, chosen from the frames the dump actually holds) and the **λ-calibrated** `φ_s^cal = min(1, λ φ_s^vor)`
  with `λ(φ_p, P) = 1 + a(P) φ_p + b(P) φ_p²` from `scripts/calibration/calibration_lambda.json`.  The calibration was
  measured on homogeneous NPT gel boxes at seven pressures (0.5–2.0), so λ needs a **local pressure per bin**.  Under
  permeation the pore pressure is not uniform — it falls from the feed to the permeate reservoir across the membrane —
  so `P_CAL_MODE='pore'` hands λ the **ramp between the measured reservoir baselines** (the same feed / permeate
  σ_zz baselines the Terzaghi split uses, per stress snapshot; Darcy with uniform k; the reservoir bins keep their own
  baseline, and λ(0, P) = 1 leaves them untouched anyway).  Panel 6(c) shows that profile, 6(d) the λ it produces.
  The alternatives are `'const'` (P_CAL everywhere — what the compression notebooks use, correct for a drained
  equilibrium where p_pore = P_target throughout the gel) and `'thermo'` (the local P_th = −⅓ tr σ^t of the nearest
  stress snapshot, which includes the network's share and is noisier).  At `dP = 0.1` the choice hardly matters:
  `a(P)` is flat around P = 1.5 and a 0.1 shift in P moves `φ_s^cal` by ≤ 0.0015 at φ_p = 0.55–0.80, below the per-bin
  CI; it matters at larger dP or at lower P, where a(P) steepens (0.54 at P = 0.5).  Applying an equilibrium calibration
  to a flowing state assumes local equilibrium in each bin (fine at dP = 0.1).  In φ_p the quadratic extrapolates outside
  the calibration's data window (φ_p^vor ≈ 0.55–0.80), pinned to the anchor; check panel 6(b) against that window.
  Method notes: `calibration_method_notes.docx` (Desktop, 2026-09-23) and `simulations/calibration_sweep/README.md`.
* **Permeability**.  `k = Q_perm L /(A ΔP)` with `L` the Rg-based gel thickness over the steady window and
  `A = l_x l_y`; reported with the **applied** `ΔP` (= `dp_piston`) and with the `ΔP` **measured** on the
  pistons (`F_fluid/A`, block-bootstrapped).  The CI combines the relative CI of `Q` and of `ΔP_meas` in
  quadrature.  In LJ units `k` is `σ⁵/(ε τ)`; it equals the `κ = k/η` of the compression notebooks (η = 1 in
  these units is NOT assumed — compare `κ = D_c/M` from the compression sweep with this `k` to test Darcy
  consistency).
* **Pressures**.  `P_feed_meas`, `P_perm_meas` are the pair forces of the mobile atoms on each sheet over
  `l_x l_y` (compute group/group; NOT `reduce sum fz`, which the aveforce fix zeroes).  Figure 1b draws them
  from `piston_force_avg` (block means over each `volume_freq` window) when that file exists, the instantaneous
  `piston_pressure` samples otherwise; the applied values are interpolated onto the block steps.  They must
  average to the applied values; the reservoir virial pressures (`pressure_feed`, `pressure_permeate`, dotted)
  are the independent check — written at the pf cadence (one block mean per `volume_freq`) since 2026-09-23,
  ten points at the stress cadence in older runs.  Thermo `press` is meaningless in this geometry (vacuum
  margins in the box).
* **Piston damper (2026-09-23)**.  The deck keeps the viscous term `−C_pist·v` on both sheets for the settle and
  the reference window only (`damp_prod=0`).  With it in production the 2026-09-22 run measured
  `P_feed − P_perm ≈ 0.03` for an applied `dP = 0.1`: the drag `C·v·N/A ≈ 0.037` per sheet ate two thirds of the
  drive, so the *applied*-dP permeability of that run is ~3× too low and only the *measured*-dP value is
  meaningful.  Runs from 2026-09-23 on have no damper in Phase 2 (the gel's hydraulic resistance alone damps
  the sheets at ~0.8 of critical).
* **Profiles**.  z-binning covers the full box (bins in the vacuum read 0).  The pore baseline for the network
  stress is the feed-reservoir interior; `pore_perm` (the permeate side) is also stored in `P['stress'][comp]`.
  **Both windows track the pistons (2026-09-22)**: they are rebuilt per stress snapshot from the measured
  `z_feed` / `z_perm` (`piston_position_…dat`), keeping every bin **entirely ≥ `res_wall_margin` (3 σ) clear of
  the piston sheet** (the bin touching a piston is depleted/layered and half its wall virial sits on the piston
  atoms, so it under-reads σ_zz by ~0.07) and ≥ 2 bins away from the gel.  The feed window therefore shrinks as
  the feed piston descends; when it would be empty the clearance is halved, and `P['bw']`, `P['bw_perm']` hold
  the masks.  The support is solvent-permeable and needs no clearance.  The summary line *plates over the run*
  reports the start → end plane of the support and both pistons.
* **Halts**.  The deck stops when the feed reservoir thins to `feed_halt_thick` or a piston face comes within
  2 σ of a box face; the run then ends cleanly (files are complete).